# Testing new graph.

In [1]:
import spacy
import nltk
import glob
import random
from pathlib import Path
import tqdm
import torch
from src.data.preprocessing import preprocessing_imdb

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

nlp = spacy.load('en_core_web_lg')


/home/trdp/Software/anaconda3/envs/phd_env/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_lg' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.0). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [2]:
# Carregar os documentos
DATASET_PATH = "../data/datasets/IMDB/train/raw/@SENTIMENT/*.txt"
positive_imdb = glob.glob(DATASET_PATH.replace("@SENTIMENT", "positive"))
negative_imdb = glob.glob(DATASET_PATH.replace("@SENTIMENT", "negative"))

positive_raw_dataset = [open(f).read() for f in tqdm.tqdm(positive_imdb, desc="Reading positive reviews")]
negative_raw_dataset = [open(f).read() for f in tqdm.tqdm(negative_imdb, desc="Reading negative reviews")]

# Option 2: List of tuples (review_text, label)
dataset = [(review, "positive") for review in positive_raw_dataset] + \
          [(review, "negative") for review in negative_raw_dataset]

random.seed(42)
random.shuffle(dataset)

text, label = dataset[0]

Reading negative reviews: 100%|██████████| 20000/20000 [00:00<00:00, 44286.43it/s]


In [3]:
preprocessed_sample = preprocessing_imdb(text, nlp)

In [4]:
print(preprocessed_sample)

There is no doubt that Halloween is by far one of the best films ever not only in its genre but also outside.I love the films creepy atmosphere like the whole it could happen here sort of situation makes it scary to think about.Also to imagine if you were ever in this situation what would you do.This is a movie that i enjoy watching highly, especially around Halloween time.John Carpenter is a very professional directer i love a lot of his other films, but there is no doubt that his best known movie is the film Halloween.Oh and if your thinking about watching the Rob Zombie remake don't.It is pure crap and a true Halloween fan would like the 1978 John Carpenter version better.Michael Myers is one of the coolest slasher killers in any film, and is a very well known one.So by all means go see this masterpiece you will really like it.


# Convert into graph

### Requirements

- input: text : str
- output: networkx digraph.
- Dependency parsing (spacy: en_core_web_lg
- Node features are based on Glove embeddings (suppose gensim)
- PMI to connect among words in a phrase.
- Sequential connection among phrases based on root node.

In [5]:
import gensim
from src.data.text_graph_dataset_parsers import GloveEmbedding
import spacy
import networkx as nx
import gensim.downloader as api
from collections import defaultdict
import itertools
import numpy as np
from math import log

# -----------------------------
# Load NLP model and embeddings
# -----------------------------
glove_vectors = gensim.models.KeyedVectors.load("../data/external/embeddings/glove_legal_100.bin", mmap='r')


[nltk_data] Downloading package punkt to /home/trdp/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
INFO:gensim.utils:loading KeyedVectors object from ../data/external/embeddings/glove_legal_100.bin
INFO:gensim.utils:loading vectors from ../data/external/embeddings/glove_legal_100.bin.vectors.npy with mmap=r
INFO:gensim.utils:KeyedVectors lifecycle event {'fname': '../data/external/embeddings/glove_legal_100.bin', 'datetime': '2025-11-12T10:18:24.416950', 'gensim': '4.3.3', 'python': '3.10.13 | packaged by conda-forge | (main, Dec 23 2023, 15:36:39) [GCC 12.3.0]', 'platform': 'Linux-6.8.0-85-generic-x86_64-with-glibc2.39', 'event': 'loaded'}


In [6]:
torch.tensor(glove_vectors["<unk>"])

tensor([-2.3435e-01, -3.4681e-02, -6.9471e-02,  4.0384e-02,  1.6417e-01,
        -5.0459e-01, -1.0660e-03, -1.4840e-01, -2.1781e-01,  1.3804e-01,
         1.0259e-01,  9.9410e-02, -1.1526e-01,  5.7564e-02,  4.5288e-02,
        -5.8340e-03,  9.3724e-02, -6.3070e-01,  1.8061e-01, -1.3939e-01,
        -2.4186e-01, -4.4134e-02, -1.9410e-02,  1.4315e-01, -6.6491e-02,
         3.8775e-02, -1.1400e-03, -2.0084e-02, -1.1127e-01, -1.7416e-01,
        -1.1256e-01, -8.5272e-02,  2.4527e-01,  8.3031e-02,  5.6871e-02,
         1.5694e-01, -7.6635e-01, -1.2101e-01, -6.9240e-03,  3.0036e-01,
         5.1700e-03, -2.3009e-02, -4.6950e-01, -1.8994e-01, -5.2931e-02,
        -1.1938e-01,  7.5762e-02, -1.9460e-03,  1.3414e-01, -1.8526e-01,
        -2.0424e+00,  6.8653e-02, -3.5185e-01,  3.5882e-01,  2.3422e-01,
         4.5734e-01,  3.2773e-02, -1.8997e-02, -1.5473e-02,  2.5370e-01,
         5.9919e-02,  7.7825e-02,  6.9114e-02, -6.1076e-01,  8.1117e-02,
        -1.2770e-01, -4.3089e-01,  8.2794e-02,  1.0

In [7]:
import tqdm
import numpy as np
import networkx as nx
from collections import defaultdict
from math import log

mean_vec = np.mean(glove_vectors.vectors, axis=0)

def get_embedding(token: str):
    token = token.lower().strip()
    if token in glove_vectors:
        return torch.tensor(glove_vectors[token])
    elif "<unk>" in glove_vectors:
        return torch.tensor(glove_vectors["<unk>"])
    else:
        return torch.tensor(mean_vec)


def compute_pmi(corpus, window_size=2):
    co_occur, word_count = defaultdict(int), defaultdict(int)
    total_windows = 0

    for phrase in corpus:
        for i, w1 in enumerate(phrase):
            word_count[w1] += 1
            window = phrase[i+1:i+1+window_size]
            for w2 in window:
                co_occur[(w1, w2)] += 1
            total_windows += len(window)

    pmi_dict = {}
    for (w1, w2), count in co_occur.items():
        p_w1 = word_count[w1] / total_windows
        p_w2 = word_count[w2] / total_windows
        p_w1w2 = count / total_windows
        pmi = log(p_w1w2 / (p_w1 * p_w2) + 1e-8)
        if pmi > 0:
            pmi_dict[(w1, w2)] = pmi
    return pmi_dict


# --- Graph builder ---
def text_to_graph(text: str) -> nx.MultiDiGraph:
    """
    Build a dependency graph (word-level) with PMI-weighted dependency edges.
    Compatible with visualization_l0().
    """
    doc = nlp(text)
    G = nx.MultiDiGraph()

    # --- Build PMI map across sentences ---
    phrase_tokens = [
        [t.text.lower().strip() for t in s if not t.is_punct and not t.is_space]
        for s in doc.sents
    ]
    pmi_dict = compute_pmi(phrase_tokens)

    # # --- Add word nodes ---
    # for token in doc:
    #     if token.is_punct or token.is_space:
    #         continue
    #
    #     tok_text = token.text.lower().strip()
    #     G.add_node(
    #         tok_text,
    #         type="word",
    #         text=tok_text,
    #         lemma=token.lemma_,
    #         pos=token.pos_,
    #         feature=get_embedding(tok_text)
    #     )

    # --- Add dependency edges (PMI-weighted) ---
    for i,token in tqdm.tqdm(enumerate(doc), desc="Adding DEP edges"):
        if token.head == token or token.is_punct or token.is_space:
            continue

        head = token.head.text.lower().strip()
        dep = token.text.lower().strip()

        G.add_node(
            f"{head}_{i}",
            type="word",
            text=head,
            lemma=token.head.lemma_,
            pos=token.head.pos_,
            feature=get_embedding(head)
        )

        G.add_node(
            dep,
            type="word",
            text=dep,
            lemma=token.lemma_,
            pos=token.pos_,
            feature=get_embedding(dep)
        )

        pmi_val = pmi_dict.get((head, dep)) or pmi_dict.get((dep, head)) or 0.0

        G.add_edge(
            head,
            dep,
            label="dep",
            weight=pmi_val,
            feature=pmi_val
        )

    # # --- Sequential edges between sentence roots ---
    # roots = [s.root.text.lower().strip() for s in doc.sents]
    # for r1, r2 in zip(roots[:-1], roots[1:]):
    #     G.add_edge(r1, r2, label="seq", weight=0.5, feature=0.5)

    return G

In [8]:
preprocessed_sample

"There is no doubt that Halloween is by far one of the best films ever not only in its genre but also outside.I love the films creepy atmosphere like the whole it could happen here sort of situation makes it scary to think about.Also to imagine if you were ever in this situation what would you do.This is a movie that i enjoy watching highly, especially around Halloween time.John Carpenter is a very professional directer i love a lot of his other films, but there is no doubt that his best known movie is the film Halloween.Oh and if your thinking about watching the Rob Zombie remake don't.It is pure crap and a true Halloween fan would like the 1978 John Carpenter version better.Michael Myers is one of the coolest slasher killers in any film, and is a very well known one.So by all means go see this masterpiece you will really like it."

In [10]:
# -----------------------------
# Example usage
# -----------------------------
graph = text_to_graph(preprocessed_sample)

print("Type", type(graph))
print("Nodes:", graph.nodes())
print("Edges:", graph.edges())

Adding DEP edges: 174it [00:00, 13483.02it/s]

Type <class 'networkx.classes.multidigraph.MultiDiGraph'>
Nodes: ['is_0', 'there', 'is', 'doubt_2', 'no', 'doubt', 'is_3', 'is_4', 'that', 'is_5', 'halloween', 'doubt_6', 'far_7', 'by', 'far', 'one_8', 'one', 'is_9', 'one_10', 'of', 'films_11', 'the', 'films', 'films_12', 'best', 'of_13', 'not_14', 'ever', 'not', 'in_15', 'in', 'not_16', 'only', 'films_17', 'genre_18', 'its', 'genre', 'in_19', 'in_20', 'but', 'but_21', 'also', 'in_22', 'outside', 'love_24', 'i', 'love', 'films_26', 'love_27', 'atmosphere_28', 'creepy', 'atmosphere', 'love_29', 'atmosphere_30', 'like', 'whole_31', 'whole', 'like_32', 'happen_33', 'it', 'happen', 'happen_34', 'could', 'love_35', 'happen_36', 'here', 'of_37', 'sort', 'happen_38', 'of_39', 'situation', 'love_40', 'makes', 'scary_41', 'scary', 'makes_42', 'think_43', 'to', 'think', 'scary_44', 'think_45', 'about', 'imagine_47', 'imagine', 'imagine_48', 'were_50', 'if', 'were', 'were_51', 'you', 'do_52', 'do', 'were_53', 'were_54', 'situation_55', 'this', 'i

In [11]:
from src.multi_graph.visualization import visualize_structural_graph, visualization_l0

deps = nlp.get_pipe("parser").labels
dep_label_map = {label: idx for idx, label in enumerate(deps)}

visualization_l0(
    nx_graph=graph,
    output_filename=str("test_result.html")
)

KeyError: 'lemma'